# 40. Calling APIs — `requests · httpx · aiohttp`
**Python Libraries Lab · Phase 3 · Web & Networking**

_Talk to web services: GET, POST, headers, errors and speed._

✅ **Runs in Colab** as-is.

Most data and services live behind web APIs. To use them, your program sends an HTTP request and reads the response. The requests library made this simple and is still the most used. httpx works almost the same but also supports async. aiohttp is an async library built for sending many requests at the same time.

**In this notebook:** This notebook starts its own small practice API, so everything works offline. You will fetch products with filters, create an order with POST, send a token in a header, handle errors and timeouts, reuse connections with a session, and then call twenty slow endpoints at once with httpx and aiohttp to see async speed. A final optional step calls a real public API.

**How to use:** run each cell in order with **Shift + Enter**, or use **Runtime → Run all**. Read the short note above each cell first, then change the code and run it again: that is how you learn.

## Setup
The next cell makes sure the library is installed (Colab already has most of them; it takes a few seconds).

In [ ]:
!pip install -q requests httpx aiohttp
import importlib.metadata as md
for pkg in ['requests', 'httpx', 'aiohttp']:
    print(pkg, md.version(pkg))

## Step 1 · Start a practice API
A tiny server built with the standard library, running in the background of this notebook.

In [ ]:
import json, threading, time
from http.server import ThreadingHTTPServer, BaseHTTPRequestHandler
from urllib.parse import urlparse, parse_qs

PRODUCTS = [{"id": 1, "name": "Steel Bottle", "price": 499}, {"id": 2, "name": "Yoga Mat", "price": 1299},
            {"id": 3, "name": "Desk Lamp", "price": 899}, {"id": 4, "name": "Notebook", "price": 149}]
ORDERS = []

class ShopAPI(BaseHTTPRequestHandler):
    def _send(self, code, body):
        data = json.dumps(body).encode()
        self.send_response(code); self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(data))); self.end_headers(); self.wfile.write(data)
    def log_message(self, *args):
        pass
    def do_GET(self):
        url = urlparse(self.path); q = parse_qs(url.query)
        if url.path == "/products":
            items = PRODUCTS
            if "max_price" in q:
                items = [p for p in items if p["price"] <= int(q["max_price"][0])]
            return self._send(200, items)
        if url.path.startswith("/products/"):
            pid = int(url.path.split("/")[-1])
            match = [p for p in PRODUCTS if p["id"] == pid]
            return self._send(200, match[0]) if match else self._send(404, {"error": "not found"})
        if url.path == "/slow":
            time.sleep(float(q.get("seconds", ["1"])[0]))
            return self._send(200, {"ok": True})
        if url.path == "/secret":
            if self.headers.get("Authorization") != "Bearer demo-token":
                return self._send(401, {"error": "missing or wrong token"})
            return self._send(200, {"secret": "ShopEasy profit is up 12%"})
        self._send(404, {"error": "unknown path"})
    def do_POST(self):
        body = json.loads(self.rfile.read(int(self.headers.get("Content-Length", 0))) or b"{}")
        if self.path == "/orders":
            if "product_id" not in body:
                return self._send(422, {"error": "product_id is required"})
            order = {"order_id": 1000 + len(ORDERS) + 1, **body}
            ORDERS.append(order)
            return self._send(201, order)
        self._send(404, {"error": "unknown path"})

server = ThreadingHTTPServer(("127.0.0.1", 8077), ShopAPI)
threading.Thread(target=server.serve_forever, daemon=True).start()
BASE = "http://127.0.0.1:8077"
print("Practice API running at", BASE)

## Step 2 · GET with requests
`params` builds the query string safely. `.json()` turns the reply into Python.

In [ ]:
import requests
r = requests.get(f"{BASE}/products", params={"max_price": 900}, timeout=5)
print(r.status_code, r.headers["Content-Type"])
print(r.url)
for p in r.json():
    print(f"  {p['name']:<14} Rs.{p['price']}")

## Step 3 · POST, headers and errors
`raise_for_status()` turns 4xx/5xx codes into exceptions you can catch.

In [ ]:
r = requests.post(f"{BASE}/orders", json={"product_id": 2, "qty": 1}, timeout=5)
print("Created:", r.status_code, r.json())

for path, headers in [("/products/99", {}), ("/secret", {}), ("/secret", {"Authorization": "Bearer demo-token"})]:
    try:
        r = requests.get(BASE + path, headers=headers, timeout=5)
        r.raise_for_status()
        print("OK  ", path, r.json())
    except requests.HTTPError as e:
        print("FAIL", path, e.response.status_code, e.response.json())

## Step 4 · Timeouts and sessions
A timeout stops waiting. A `Session` reuses the connection and default headers.

In [ ]:
try:
    requests.get(f"{BASE}/slow", params={"seconds": 3}, timeout=1)
except requests.Timeout:
    print("Gave up after 1 second (timeout)")

with requests.Session() as s:
    s.headers.update({"Authorization": "Bearer demo-token", "User-Agent": "shopeasy-lab/1.0"})
    print(s.get(f"{BASE}/secret", timeout=5).json())
    print(len(s.get(f"{BASE}/products", timeout=5).json()), "products via the same session")

## Step 5 · Many requests at once with httpx (async)
20 calls that each take 0.5 s: one by one ~10 s, together ~0.5 s.

In [ ]:
import httpx, asyncio, time
t = time.perf_counter()
with httpx.Client(timeout=5) as c:
    for _ in range(4):
        c.get(f"{BASE}/slow", params={"seconds": 0.5})
print(f"4 calls one by one: {time.perf_counter() - t:.1f}s")

async def fetch_all(n):
    async with httpx.AsyncClient(timeout=10) as c:
        return await asyncio.gather(*(c.get(f"{BASE}/slow", params={"seconds": 0.5}) for _ in range(n)))
t = time.perf_counter()
res = await fetch_all(20)
print(f"20 calls together with httpx: {time.perf_counter() - t:.1f}s, all OK: {all(r.status_code == 200 for r in res)}")

## Step 6 · The same with aiohttp
aiohttp is built only for async and is very fast for large numbers of requests.

In [ ]:
import aiohttp
async def fetch_products(ids):
    async with aiohttp.ClientSession(timeout=aiohttp.ClientTimeout(total=10)) as session:
        async def one(pid):
            async with session.get(f"{BASE}/products/{pid}") as resp:
                return resp.status, await resp.json()
        return await asyncio.gather(*(one(i) for i in ids))
for status, body in await fetch_products([1, 2, 3, 4, 5]):
    print(status, body)

## Step 7 · Optional: a real public API
Calls GitHub's public API (no key needed, limited to 60 calls per hour). Skips politely if the internet is unavailable.

In [ ]:
try:
    r = requests.get("https://api.github.com/repos/pandas-dev/pandas", timeout=10)
    r.raise_for_status()
    repo = r.json()
    print(f"{repo['full_name']}: {repo['stargazers_count']:,} stars, {repo['forks_count']:,} forks, language {repo['language']}")
except requests.RequestException as e:
    print("Could not reach GitHub right now:", type(e).__name__)

## ✍️ Your turn
POST an order for `product_id` 3 with `qty` 2 to `BASE + '/orders'`, and store the **order_id** from the reply in `answer`.

Write your code in the next cell, run it, then run the check cell.

In [ ]:
answer = None  # your code here

In [ ]:
def check():
    ok = isinstance(answer, int) and answer == ORDERS[-1]["order_id"] and ORDERS[-1]["product_id"] == 3
    return ok, f"Expected the new order's id, got {answer}"

# ▶ Run this cell to check your answer
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

### Stuck?
The next cell holds a solution. In Colab its code is hidden: double-click the cell to see it, or just run it.

In [ ]:
# @title 👀 Solution (double-click to show the code) { display-mode: "form" }
answer = requests.post(f"{BASE}/orders", json={"product_id": 3, "qty": 2}, timeout=5).json()["order_id"]

# Check again with the solution
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

## Recap
- **HTTP method**
- **Status code**
- **Query params**
- **Headers / token**
- **Timeout**
- **Async client**

**Next:** 41. Parsing HTML (`beautifulsoup4 · lxml`): [open in Colab](https://colab.research.google.com/github/aashaysocial27-maker/python-libraries-colab/blob/main/notebooks/41_beautifulsoup_lxml.ipynb)

## 💻 Run it on your own computer
1. Install Python 3.10+ from python.org (Windows: tick **Add python.exe to PATH**).
2. Open a terminal (Windows: *Command Prompt*; Mac: *Terminal*) and run:
```bash
mkdir python-lab && cd python-lab
python -m venv .venv
# Windows:  .venv\Scripts\activate      Mac/Linux:  source .venv/bin/activate
pip install requests httpx aiohttp
pip install notebook
jupyter notebook
```
3. In Colab choose **File → Download → Download .ipynb**, then open that file in the Jupyter page that opened in your browser.

Lines that start with `!` or `%%` are notebook commands. In a plain terminal, type `!` commands without the `!`.